# Visualizing data granules

Many datasets provide metadata on their location as well as lightweight visualizations that we can use to quickly get an idea of what a granule contains. `earthaccess` exposes some of this metadata, letting us conveniently visualize granules in a notebook.

In this example, we will use these capabilities to make an interactive map of Landsat granules.

### Get some granules

In [ ]:
import earthaccess
import geopandas as gpd
import pandas as pd
import folium

earthaccess.login()

In [ ]:
# Search for data

results = earthaccess.search_data(
    short_name='HLSL30',
    cloud_hosted=True,
    bounding_box=(-10, 20, 10, 50),
    temporal=("2019-01", "2019-03"),
    count=10
)

How can you tell if a granule supports visualization? Simply check the contents of the `dataviz_links` property. This points to the visualization we can use. For Landsat, this is a low-resolution true-color preview of the data.

In [ ]:
from IPython.display import Image

print(results[0].dataviz_links[0])

Image(url=results[0].dataviz_links[0], width=512)

### Map the granules

So we have a preview of the granule. But, we can also show where it is in the world. `earthaccess` granules inherit from Python dictionaries and implement a `__geo_interface__`, so we can easily convert them into a GeoDataFrame.

In [ ]:
results_gdf = gpd.GeoDataFrame(
    data=pd.json_normalize(results),
    geometry=gpd.GeoSeries(results, crs=4326)
)

Then, we can either plot the geometries.

In [ ]:
results_gdf.explore(tootlip="umm.GranuleUR")

Or, we can overlay the preview image for each granule onto its actual position on the map!

In [ ]:
m = folium.Map([0, 0], zoom_start=2)

# Dataviz links do not carry through on dict conversion, so we have
# to do this iteration thing.
for (idx, row), granule in zip(results_gdf.iterrows(), results):
    bbox = row["geometry"].bounds
    img_url = granule.dataviz_links[0] 
    
    img = folium.raster_layers.ImageOverlay(
        name=row["umm.GranuleUR"],
        image=img_url,
        bounds=[(bbox[1], bbox[0]), [bbox[3], bbox[2]]],
        opacity=1.0,
        interactive=True,
        cross_origin=False,
        zindex=1,
    )
    
    folium.Popup(row["umm.GranuleUR"]).add_to(img)
    
    img.add_to(m)
    
m
    

This approach works well if the visualization image can be logically placed on a map. This is not always the case! For example, ICEsat2 granules provide many plots in `dataviz_links`. It would not make sense to place these on a map.

In [ ]:
icesat_granules = earthaccess.search_data(
    short_name="ATL06",
    count=10
)

In [ ]:
Image(icesat_granules[0].dataviz_links[0], width=512)

However, we can still plot the tracks of each granule.

In [ ]:
icesat_gdf = gpd.GeoDataFrame(
    data=pd.json_normalize(icesat_granules),
    geometry=gpd.GeoSeries(icesat_granules, crs=4326)
)

icesat_gdf.explore(tooltip="umm.GranuleUR")